In [30]:
from pathlib import Path

import numpy as np
import rasterio
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
from rasterio.mask import mask


In [5]:
CLC_PATH = Path(
    "../data/raw/clc2018/DATA/U2018_CLC2018_V2020_20u1.tif"
)

print(CLC_PATH.exists())

True


In [6]:
with rasterio.open(CLC_PATH) as src:
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bands:", src.count)
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("Transform:", src.transform)
    print("NoData:", src.nodata)
    print("Data types:", src.dtypes)

Width: 65000
Height: 46000
Bands: 1
CRS: PROJCS["ETRS89-extended / LAEA Europe",GEOGCS["ETRS89",DATUM["European_Terrestrial_Reference_System_1989",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6258"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4258"]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["latitude_of_center",52],PARAMETER["longitude_of_center",10],PARAMETER["false_easting",4321000],PARAMETER["false_northing",3210000],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3035"]]
Bounds: BoundingBox(left=900000.0, bottom=900000.0, right=7400000.0, top=5500000.0)
Resolution: (100.0, 100.0)
Transform: | 100.00, 0.00, 900000.00|
| 0.00,-100.00, 5500000.00|
| 0.00, 0.00, 1.00|
NoData: -128.0
Data types: ('int8',)


In [7]:
with rasterio.open(CLC_PATH) as src:
    print("Descriptions:", src.descriptions)
    print("Dataset tags:", src.tags())
    print("Band tags:", src.tags(1))

Descriptions: (None,)
Dataset tags: {'AREA_OR_POINT': 'Area', 'DataType': 'Thematic'}
Band tags: {'RepresentationType': 'THEMATIC', 'STATISTICS_COVARIANCES': '136.429646247598', 'STATISTICS_MAXIMUM': '48', 'STATISTICS_MEAN': '25.753373398066', 'STATISTICS_MINIMUM': '1', 'STATISTICS_SKIPFACTORX': '1', 'STATISTICS_SKIPFACTORY': '1', 'STATISTICS_STDDEV': '11.680310194836'}


In [11]:
with rasterio.open(CLC_PATH) as src:
    print("Color interpretation:", src.colorinterp)
    

Color interpretation: (<ColorInterp.gray: 1>,)


In [16]:
VAT_PATH = VAT_PATH = CLC_PATH.parent / "U2018_CLC2018_V2020_20u1.tif.vat.dbf"

print(VAT_PATH)
print(VAT_PATH.exists())

..\data\raw\clc2018\DATA\U2018_CLC2018_V2020_20u1.tif.vat.dbf
True


In [19]:
vat = gpd.read_file(VAT_PATH)

print(vat.columns)
print(vat.head(10))

Index(['Value', 'Count', 'LABEL3', 'Red', 'Green', 'Blue', 'CODE_18'], dtype='str')
   Value       Count                                      LABEL3       Red  \
0      1    800699.0                     Continuous urban fabric  0.901961   
1      2  17085234.0                  Discontinuous urban fabric  1.000000   
2      3   3210212.0              Industrial or commercial units  0.800000   
3      4    414626.0  Road and rail networks and associated land  0.800000   
4      5    122685.0                                  Port areas  0.901961   
5      6    352020.0                                    Airports  0.901961   
6      7    820443.0                    Mineral extraction sites  0.650980   
7      8    125373.0                                  Dump sites  0.650980   
8      9    201018.0                          Construction sites  1.000000   
9     10    330596.0                           Green urban areas  1.000000   

      Green      Blue CODE_18  
0  0.000000  0.301961    

In [20]:
BW_PATH = Path("../data/processed/baden_wuerttemberg.gpkg")

bw = gpd.read_file(BW_PATH)

print(bw.crs)

EPSG:4326


In [21]:
bw_LAEA_eu = bw.to_crs(epsg=3035)

In [22]:
print(bw_LAEA_eu.crs)
print(bw_LAEA_eu.total_bounds)

EPSG:3035
[4134158.31154794 2715903.22919998 4357513.18278567 2964380.71541958]


In [24]:
with rasterio.open(CLC_PATH) as src:
    bw_clc, bw_clc_transform = mask(
        src,
        bw_LAEA_eu.geometry,
        crop=True,
        filled=False
    )

In [25]:
print("Shape:", bw_clc.shape)
print("Type:", type(bw_clc))
print("Transform:")
print(bw_clc_transform)

Shape: (1, 2485, 2235)
Type: <class 'numpy.ma.MaskedArray'>
Transform:
| 100.00, 0.00, 4134100.00|
| 0.00,-100.00, 2964400.00|
| 0.00, 0.00, 1.00|


In [26]:
bw_clc_2d = bw_clc[0]

In [27]:
print("2D shape:", bw_clc_2d.shape)
print("Total cells:", bw_clc_2d.size)
print("Valid cells:", bw_clc_2d.count())
print("Masked cells:", bw_clc_2d.mask.sum())

2D shape: (2485, 2235)
Total cells: 5553975
Valid cells: 3574856
Masked cells: 1979119


In [28]:
clc_values = bw_clc_2d.compressed()

In [31]:
values, counts = np.unique(
    clc_values,
    return_counts=True
)

bw_counts = pd.DataFrame({
    "Value": values,
    "Count_BW": counts
})

bw_counts.head()

,Value,Count_BW
0,1,13076
1,2,255926
2,3,61062
3,4,1360
4,5,1358


In [32]:
vat.head()

,Value,Count,LABEL3,Red,Green,Blue,CODE_18
0,1,800699.0,Continuous urban fabric,0.901961,0.000000,0.301961,111
1,2,17085234.0,Discontinuous urban fabric,1.000000,0.000000,0.000000,112
2,3,3210212.0,Industrial or commercial units,0.800000,0.301961,0.949020,121
3,4,414626.0,Road and rail networks and associated land,0.800000,0.000000,0.000000,122
4,5,122685.0,Port areas,0.901961,0.800000,0.800000,123


In [33]:
bw_clc_vat = pd.merge(bw_counts,vat, on='Value', how='left')
bw_clc_vat.head()

,Value,Count_BW,Count,LABEL3,Red,Green,Blue,CODE_18
0,1,13076,800699.0,Continuous urban fabric,0.901961,0.000000,0.301961,111
1,2,255926,17085234.0,Discontinuous urban fabric,1.000000,0.000000,0.000000,112
2,3,61062,3210212.0,Industrial or commercial units,0.800000,0.301961,0.949020,121
3,4,1360,414626.0,Road and rail networks and associated land,0.800000,0.000000,0.000000,122
4,5,1358,122685.0,Port areas,0.901961,0.800000,0.800000,123


In [34]:
bw_clc_vat["Percentage"] = (bw_clc_vat["Count_BW"] / bw_clc_2d.count()) * 100

In [36]:
cols_to_keep = ["CODE_18", "LABEL3", "Count_BW", "Percentage"]

In [37]:
bw_clc_df = bw_clc_vat[cols_to_keep].copy()

In [40]:
bw_clc_df = bw_clc_df.sort_values(by="Count_BW", ascending=False)
bw_clc_df.head(10)

,CODE_18,LABEL3,Count_BW,Percentage
11,211,Non-irrigated arable land,996913,27.886802
18,312,Coniferous forest,637092,17.821473
14,231,Pastures,610459,17.076464
17,311,Broad-leaved forest,548238,15.335946
1,112,Discontinuous urban fabric,255926,7.159058
19,313,Mixed forest,205324,5.743560
13,222,Fruit trees and berry plantations,101854,2.849178
2,121,Industrial or commercial units,61062,1.708097
12,221,Vineyards,37667,1.053665
22,324,Transitional woodland-shrub,23663,0.661929


In [41]:
bw_clc_df["Percentage"].sum()

np.float64(100.0)

In [42]:
bw_clc_df["Area_km2"] = bw_clc_df["Count_BW"] / 100 
bw_clc_df.head(5)

,CODE_18,LABEL3,Count_BW,Percentage,Area_km2
11,211,Non-irrigated arable land,996913,27.886802,9969.13
18,312,Coniferous forest,637092,17.821473,6370.92
14,231,Pastures,610459,17.076464,6104.59
17,311,Broad-leaved forest,548238,15.335946,5482.38
1,112,Discontinuous urban fabric,255926,7.159058,2559.26


In [ ]:
bw_clc_df["Area_km2"].sum() #Sanity check: The total area of Baden-Württemberg is approximately 35,751 km², which is consistent with the calculated area from the land cover data.

np.float64(35748.56)